# Expert Q-values with FrozenLake

`q_star_source` is an opt-in field on `EnvConfig` that attaches expert Q-values to every step output as `outputs[i]["q_star"]`. Once configured, the array is available at every step — including the very first reset frame — so a rollout can choose greedy expert actions without training a separate policy.

## Why attach Q-values at the env level?

Attaching Q* at rollout time rather than post-hoc keeps training code clean: the expert is just another field in the output dict, treated the same way as `reward` or `observation`. It is useful for:

- **Imitation and distillation** — supervise a learned policy against `argmax(q_star)` without a separate data pipeline.
- **Diagnostics** — compare the agent's value estimates against exact Q-values to measure how far the learned policy has drifted from optimal.
- **Guided exploration** — epsilon-greedy or softmax over `q_star` to bias rollouts toward useful states without hardcoding a policy.

## The two `q_star_source` providers

`q_star_source` is a plain dict with a `"provider"` key. Two providers are available:

| Provider | When to use |
|----------|-------------|
| `"hf_q_table"` | You have a precomputed Q-table stored as a pickle file (locally or on the Hugging Face Hub). Pass `"path"` for a local file or `"repo_id"` + `"filename"` to download. The file must contain `{"qtable": np.ndarray[states, actions]}`. |
| `"metadata_q_star"` | The env computes Q* itself — used by `SyntheticEnv-v1` and `Procedural-FrozenLake-v1`. No file required; the env runs value iteration internally for each new map. |

This notebook uses `FrozenLake-v1` (fixed 4×4 map, slippery) with the `hf_q_table` provider. Q-values are computed offline by **value iteration** and saved to a local pickle file, then loaded at env-build time.

## Imports

In [ ]:
import pickle
import tempfile

import gymnasium as gym
import numpy as np
import torch

from mouse_envs import EnvConfig, make_env

## Compute Q* via value iteration

`FrozenLake-v1` is a small finite MDP (16 states × 4 actions). Value iteration converges to the exact optimal Q-table in a few hundred iterations. `env.unwrapped.P[s][a]` gives the transition list `(prob, next_state, reward, done)` directly from Gymnasium.

In [ ]:
fl = gym.make("FrozenLake-v1", is_slippery=True)
base = fl.unwrapped
n_states  = base.observation_space.n  # 16
n_actions = base.action_space.n       # 4
fl.close()

gamma = 0.99
V = np.zeros(n_states)
for _ in range(1000):
    Q = np.zeros((n_states, n_actions))
    for s in range(n_states):
        for a in range(n_actions):
            for prob, next_s, reward, done in base.P[s][a]:
                Q[s, a] += prob * (reward + gamma * V[next_s] * (not done))
    V = Q.max(axis=1)

print("Q* (first 4 states):")
for s in range(4):
    print(f"  s={s}  {Q[s].round(3)}  greedy_action={int(np.argmax(Q[s]))}")

## Save Q-table and build the environment

The `hf_q_table` provider reads a pickle file containing `{"qtable": np.ndarray[states, actions]}`. Passing a local `path` skips any Hugging Face download.

In [ ]:
tmp = tempfile.NamedTemporaryFile(suffix=".pkl", delete=False)
pickle.dump({"qtable": Q}, tmp)
tmp.flush()
qtable_path = tmp.name

cfg = EnvConfig(
    id="FrozenLake-v1",
    seed=0,
    num_envs=1,
    max_episode_steps=100,
    kwargs={"is_slippery": True},
    q_star_source={"provider": "hf_q_table", "path": qtable_path},
)
env = make_env(cfg)

## Inspect the reset frame

The first `step` performs the internal reset. `q_star` is already attached — the 4 Q-values for the start state.

In [ ]:
outputs = env.step(env.sample_random_inputs())

print(f"observation (state index): {outputs[0]['observation'].item()}")
print(f"q_star:                    {outputs[0]['q_star'].round(3)}")
print(f"greedy action:             {int(np.argmax(outputs[0]['q_star']))}")

## Expert rollout

Take `argmax` over `q_star` each step. Because FrozenLake is slippery, the agent does not always move where intended, so episodes may still fail — but the greedy policy is optimal in expectation.

In [ ]:
episodes = 0

for step in range(300):
    inputs = [{
        "action": torch.tensor(int(np.argmax(outputs[0]["q_star"])), dtype=torch.int64)
    }]
    outputs = env.step(inputs)

    r = outputs[0]
    if r["done"].item() != 0:
        episodes += 1
        outcome = "reached goal" if r["reward"].item() > 0 else "fell in hole"
        print(f"step={step:3d}  episode={episodes}  {outcome}")

env.close()